In [4]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

############################################
# 0. 配置
############################################
train_path_rent = "副本ruc_Class25Q2_train_rent91.csv"  # 有Price
test_path_rent  = "要用租房的数据.csv"                       # 无Price, 要预测
TARGET_COL      = "Price"

K_CLUSTERS = 80  # 我们还是用80类市场聚类

# 下面四个超参请用“租房版”的最佳一组
RIDGE_ALPHA_BEST   = 100.0
LASSO_ALPHA_BEST   = 1000.0
ENET_ALPHA_BEST    = 10.0
ENET_L1RATIO_BEST  = 0.999

OUT_OLS_XLS        = "rent_pred_OLS.xlsx"
OUT_RIDGE_XLS      = "rent_pred_Ridge.xlsx"
OUT_LASSO_XLS      = "rent_pred_Lasso.xlsx"
OUT_ENET_XLS       = "rent_pred_ElasticNet.xlsx"

############################################
# 1. 读数据
############################################
df_tr_raw = pd.read_csv(train_path_rent)
df_te_raw = pd.read_csv(test_path_rent)

# 如果测试集没有 ID 那就造一个
if "ID" not in df_te_raw.columns:
    df_te_raw["ID"] = np.arange(len(df_te_raw)) + 1_000_000

df_te_idonly = df_te_raw[["ID"]].copy()

############################################
# 2. 数值化 + 交互项
############################################
# 注意：下面这些列名必须和你的训练/测试文件完全一致
# 如果训练集里是 "绿 化 率" (有空格)，把 "绿化率" 改成 "绿 化 率"。
# 同理 "容 积 率","物 业 费" 等等。
numeric_cols = [
    TARGET_COL,           # 训练集才有
    "面积", "房龄", "室",
    "房屋总数", "楼栋总数", "绿化率", "容积率", "物业费",
    "lon", "lat",
    "楼层_中层", "楼层_高层", "楼层_地下室",
    "电梯_有",
    "租赁方式_整租",
    # 付款方式
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    # 朝向
    "朝向_东","朝向_南","朝向_西","朝向_北",
    # 年 / 月 dummy
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    # 设施
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
]

def force_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    # 电梯_有：缺失就当0
    if "电梯_有" in out.columns:
        out["电梯_有"] = out["电梯_有"].fillna(0)
    return out

df_tr_num = force_numeric(df_tr_raw, numeric_cols)
df_te_num = force_numeric(df_te_raw, numeric_cols)

def add_interactions(df):
    out = df.copy()
    # 面积×高层
    if "面积" in out.columns and "楼层_高层" in out.columns:
        out["交互_面积x高层"] = (
            pd.to_numeric(out["面积"], errors="coerce")
            * pd.to_numeric(out["楼层_高层"], errors="coerce")
        )
    # 高层×电梯有
    if "楼层_高层" in out.columns and "电梯_有" in out.columns:
        out["交互_高层x电梯有"] = (
            pd.to_numeric(out["楼层_高层"], errors="coerce")
            * pd.to_numeric(out["电梯_有"], errors="coerce")
        )
    # 面积×房龄
    if "面积" in out.columns and "房龄" in out.columns:
        out["交互_面积x房龄"] = (
            pd.to_numeric(out["面积"], errors="coerce")
            * pd.to_numeric(out["房龄"], errors="coerce")
        )
    return out

df_tr_num = add_interactions(df_tr_num)
df_te_num = add_interactions(df_te_num)

############################################
# 3. KMeans聚类 (market_cluster80)
#    直接对每一套房做聚类，不用 '板块'，不用 'subway'
#    只能用训练、测试都具备的公共环境指标
############################################
cluster_basis_cols = [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "电梯_有",
]

# 准备训练集聚类矩阵
train_cluster_mat = df_tr_num[cluster_basis_cols].copy()
for c in train_cluster_mat.columns:
    train_cluster_mat[c] = pd.to_numeric(train_cluster_mat[c], errors="coerce")
train_cluster_mat = train_cluster_mat.fillna(train_cluster_mat.median(numeric_only=True))

# 标准化并聚类
scaler_cluster = StandardScaler()
train_scaled = scaler_cluster.fit_transform(train_cluster_mat)

K_use = min(K_CLUSTERS, len(df_tr_num))
kmeans_model = KMeans(n_clusters=K_use, random_state=42, n_init=10)
df_tr_num["market_cluster80"] = kmeans_model.fit_predict(train_scaled).astype(int)

# 测试集同样打标签
test_cluster_mat = df_te_num[cluster_basis_cols].copy()
for c in test_cluster_mat.columns:
    test_cluster_mat[c] = pd.to_numeric(test_cluster_mat[c], errors="coerce")

# 用训练中位数来填测试缺失
test_cluster_mat = test_cluster_mat.fillna(train_cluster_mat.median(numeric_only=True))

test_scaled = scaler_cluster.transform(test_cluster_mat)
df_te_num["market_cluster80"] = kmeans_model.predict(test_scaled).astype(int)

############################################
# 4. 构造特征矩阵 X_train / X_test
#    加城市dummy、cluster80 dummy、交互项
############################################

# 我们最终想喂回归模型的特征（租金版）
base_feature_cols = [
    "面积","房龄","室",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "租赁方式_整租",
    # 付款方式
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    # 朝向
    "朝向_东","朝向_南","朝向_西","朝向_北",
    # 时间
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7",
    "月_8","月_9","月_10","月_11","月_12",
    # 设施
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    # 交互
    "交互_面积x高层","交互_高层x电梯有","交互_面积x房龄",
]

def build_design_matrix(df):

    df2 = df.copy()

    # 1) 城市dummy
    #    注意我们不会 drop_first，因为后面还会去掉常数列
    if "城市" in df2.columns:
        city_dum = pd.get_dummies(df2["城市"], prefix="city", drop_first=False).astype(int)
    else:
        city_dum = pd.DataFrame(index=df2.index)

    # 2) cluster80 dummy
    clus_dum = pd.get_dummies(
        df2["market_cluster80"],
        prefix="cluster80",
        drop_first=False
    ).astype(int)

    # 3) 主特征
    keep_cols = [c for c in base_feature_cols if c in df2.columns]
    X_main = df2[keep_cols].copy()
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    # 4) 合并
    X_full_local = pd.concat(
        [
            X_main.reset_index(drop=True),
            city_dum.reset_index(drop=True),
            clus_dum.reset_index(drop=True),
        ],
        axis=1
    )

    # 5) 去掉标准差=0的列（常数列）
    X_full_local = X_full_local.loc[:, X_full_local.std(axis=0) > 0]

    return X_full_local

X_tr_full = build_design_matrix(df_tr_num)
X_te_full = build_design_matrix(df_te_num)

# 让测试集和训练集列完全一致
train_cols = X_tr_full.columns
X_te_full = X_te_full.reindex(columns=train_cols, fill_value=0)

############################################
# 5. y + IQR 去极端值 (训练)
############################################
y_tr_raw = pd.to_numeric(df_tr_num[TARGET_COL], errors="coerce")
y_tr_raw = y_tr_raw.fillna(y_tr_raw.median())

Q1 = y_tr_raw.quantile(0.25)
Q3 = y_tr_raw.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR

mask_ok = (y_tr_raw >= lower_cut) & (y_tr_raw <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr_raw.loc[mask_ok].reset_index(drop=True)

X_test_final = X_te_full.reset_index(drop=True)

print("训练样本(去极端后):", X_ok.shape)
print("测试样本:", X_test_final.shape)

############################################
# 6. 定义4个模型（OLS, Ridge, Lasso, ENet）
############################################
from sklearn.preprocessing import StandardScaler

model_ols = make_pipeline(
    StandardScaler(with_mean=False),
    LinearRegression()
)

model_ridge = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(
        alpha=RIDGE_ALPHA_BEST,
        fit_intercept=True,
        random_state=42
    )
)

model_lasso = make_pipeline(
    StandardScaler(with_mean=False),
    Lasso(
        alpha=LASSO_ALPHA_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

model_enet = make_pipeline(
    StandardScaler(with_mean=False),
    ElasticNet(
        alpha=ENET_ALPHA_BEST,
        l1_ratio=ENET_L1RATIO_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

############################################
# 7. 拟合模型（用过滤后的训练集）
############################################
print("拟合 OLS...")
model_ols.fit(X_ok, y_ok)

print("拟合 Ridge...")
model_ridge.fit(X_ok, y_ok)

print("拟合 Lasso...")
model_lasso.fit(X_ok, y_ok)

print("拟合 ElasticNet...")
model_enet.fit(X_ok, y_ok)

############################################
# 8. 对测试集做预测
############################################
pred_ols   = model_ols.predict(X_test_final)
pred_ridge = model_ridge.predict(X_test_final)
pred_lasso = model_lasso.predict(X_test_final)
pred_enet  = model_enet.predict(X_test_final)

############################################
# 9. 输出 (ID, Price) 两列，写Excel/CSV
############################################
sub_ols = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_ols, 0).astype(int)
})
sub_ridge = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_ridge, 0).astype(int)
})
sub_lasso = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_lasso, 0).astype(int)
})
sub_enet = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_enet, 0).astype(int)
})

sub_ols.to_excel(OUT_OLS_XLS, index=False)
sub_ridge.to_excel(OUT_RIDGE_XLS, index=False)
sub_lasso.to_excel(OUT_LASSO_XLS, index=False)
sub_enet.to_excel(OUT_ENET_XLS, index=False)

# 如果要CSV
sub_ols.to_csv("rent_pred_OLS.csv", index=False)
sub_ridge.to_csv("rent_pred_Ridge.csv", index=False)
sub_lasso.to_csv("rent_pred_Lasso.csv", index=False)
sub_enet.to_csv("rent_pred_ElasticNet.csv", index=False)

print("导出完成：(ID, Price) for all 4 models ✅")



训练样本(去极端后): (93365, 140)
测试样本: (9773, 140)
拟合 OLS...
拟合 Ridge...
拟合 Lasso...
拟合 ElasticNet...
导出完成：(ID, Price) for all 4 models ✅


In [7]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

############################################
# 0. 配置区（你按自己情况改这几行）
############################################
train_path = "副本ruc_Class25Q2_train_rent91.csv"   # 训练集（有Price）
test_path  = "要用租房的数据.csv"                     # 测试集（要预测）

TARGET_COL = "Price"
K_CLUSTERS = 80   # 我们想分80类市场段位

# 你的调参结果（租金版的最优值，先放一组示例，按你自己改）
RIDGE_ALPHA_BEST   = 100.0
LASSO_ALPHA_BEST   = 1000.0
ENET_ALPHA_BEST    = 10.0
ENET_L1RATIO_BEST  = 0.9

# 输出文件名（两列：ID, Price）
OUT_OLS  = "rent_pred_OLS.xlsx"
OUT_RID  = "rent_pred_Ridge.xlsx"
OUT_LAS  = "rent_pred_Lasso.xlsx"
OUT_ENET = "rent_pred_ElasticNet.xlsx"

############################################
# 1. 读数据
############################################
df_tr_raw = pd.read_csv(train_path)
df_te_raw = pd.read_csv(test_path)

# 如果测试集没有 ID，就造一个
if "ID" not in df_te_raw.columns:
    df_te_raw["ID"] = np.arange(len(df_te_raw)) + 1_000_000

# 有些列在不同导出版本里可能带奇怪空格，这里提前标准化一下列名（如果你确认两边完全一致，也可以不做）
rename_map = {
    "绿 化 率": "绿化率",
    "容 积 率": "容积率",
    "物 业 费": "物业费",
    # 如果还有别的“有空格版本”，也在这里补
}
df_tr_raw = df_tr_raw.rename(columns=rename_map)
df_te_raw = df_te_raw.rename(columns=rename_map)

############################################
# 2. 基础数值化 + 交互项
############################################
# 根据我们实际看到的列，训练集有这些列：
# 面积, 室, 城市, Price, lon, lat, 房屋总数, 楼栋总数, 绿化率, 容积率, 物业费,
# 朝向_东, 朝向_南, 朝向_西, 朝向_北,
# 年_2025, 月_1, 月_2, 月_3, 月_4, 月_6, 月_7, 月_8, 月_9, 月_10, 月_11, 月_12,
# 楼层_中层, 楼层_高层, 楼层_地下室, 电梯_有,
# 付款方式_双月付价, 付款方式_季付价, 付款方式_年付价, 付款方式_月付价,
# 租赁方式_整租,
# 房龄,
# 设施_床, 设施_衣柜, 设施_空调, 设施_洗衣机, 设施_热水器, 设施_冰箱,
# 设施_天然气, 设施_电视, 设施_暖气, 设施_宽带
#
# 测试集还多了一个 "板块"，训练集没有；我们可以不用它训练没关系。

numeric_cols = [
    TARGET_COL,  # 训练集用，测试集里这列是没有的，后面会自动忽略
    "面积","室","lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室","电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7",
    "月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
]

def force_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    # 电梯_有 缺失当0
    if "电梯_有" in out.columns:
        out["电梯_有"] = out["电梯_有"].fillna(0)
    return out

df_tr_num = force_numeric(df_tr_raw, numeric_cols)
df_te_num = force_numeric(df_te_raw, numeric_cols)

# 交互项：面积×高层, 高层×电梯有, 面积×房龄
def add_interactions(df):
    out = df.copy()
    if "面积" in out.columns and "楼层_高层" in out.columns:
        out["交互_面积x高层"] = (
            pd.to_numeric(out["面积"], errors="coerce")
            * pd.to_numeric(out["楼层_高层"], errors="coerce")
        )
    if "楼层_高层" in out.columns and "电梯_有" in out.columns:
        out["交互_高层x电梯有"] = (
            pd.to_numeric(out["楼层_高层"], errors="coerce")
            * pd.to_numeric(out["电梯_有"], errors="coerce")
        )
    if "面积" in out.columns and "房龄" in out.columns:
        out["交互_面积x房龄"] = (
            pd.to_numeric(out["面积"], errors="coerce")
            * pd.to_numeric(out["房龄"], errors="coerce")
        )
    return out

df_tr_num = add_interactions(df_tr_num)
df_te_num = add_interactions(df_te_num)

############################################
# 3. 用公共物理特征聚类 KMeans → market_cluster80
#    注意：不能用 Price，因为测试集没有 Price
############################################
cluster_basis_cols = [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "电梯_有",
]

# 训练集聚类数据
train_cluster_mat = df_tr_num[cluster_basis_cols].copy()
for c in train_cluster_mat.columns:
    train_cluster_mat[c] = pd.to_numeric(train_cluster_mat[c], errors="coerce")
train_cluster_mat = train_cluster_mat.fillna(train_cluster_mat.median(numeric_only=True))

# 统一标准化再聚类
scaler_cluster = StandardScaler()
train_scaled = scaler_cluster.fit_transform(train_cluster_mat)

K_use = min(K_CLUSTERS, len(df_tr_num))
kmeans_model = KMeans(n_clusters=K_use, random_state=42, n_init=10)
df_tr_num["market_cluster80"] = kmeans_model.fit_predict(train_scaled).astype(int)

# 测试集同样打标签
test_cluster_mat = df_te_num[cluster_basis_cols].copy()
for c in test_cluster_mat.columns:
    test_cluster_mat[c] = pd.to_numeric(test_cluster_mat[c], errors="coerce")
# 用训练集的中位数补测试的缺失
test_cluster_mat = test_cluster_mat.fillna(train_cluster_mat.median(numeric_only=True))
test_scaled = scaler_cluster.transform(test_cluster_mat)
df_te_num["market_cluster80"] = kmeans_model.predict(test_scaled).astype(int)

############################################
# 4. 组最终的特征矩阵 X_tr_full / X_te_full
#    内容：
#    - 主特征 (面积、房龄、楼层、电梯、付款方式、朝向、时间、设施…)
#    - 城市 dummy
#    - cluster80 dummy
#    - 交互项
############################################
base_feature_cols = [
    "面积","室","房龄",
    "lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7",
    "月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x高层","交互_高层x电梯有","交互_面积x房龄",
]

def build_design_matrix(df):
    df2 = df.copy()

    # 1. 城市 dummy
    if "城市" in df2.columns:
        city_dum = pd.get_dummies(
            df2["城市"],
            prefix="city",
            drop_first=False
        ).astype(int)
    else:
        city_dum = pd.DataFrame(index=df2.index)

    # 2. cluster80 dummy
    clus_dum = pd.get_dummies(
        df2["market_cluster80"],
        prefix="cluster80",
        drop_first=False
    ).astype(int)

    # 3. 主特征
    keep_cols = [c for c in base_feature_cols if c in df2.columns]
    X_main = df2[keep_cols].copy()
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    # 4. 合并成一个大矩阵
    X_full_local = pd.concat(
        [
            X_main.reset_index(drop=True),
            city_dum.reset_index(drop=True),
            clus_dum.reset_index(drop=True),
        ],
        axis=1
    )

    # 5. 去掉标准差为0的列（常数列）
    X_full_local = X_full_local.loc[:, X_full_local.std(axis=0) > 0]

    return X_full_local

X_tr_full = build_design_matrix(df_tr_num)
X_te_full = build_design_matrix(df_te_num)

# 用训练集的列顺序来对齐测试集
train_cols = X_tr_full.columns
X_te_full = X_te_full.reindex(columns=train_cols, fill_value=0)

# 最终喂给模型的测试特征矩阵
X_test_final = X_te_full.copy()

############################################
# 5. y & IQR过滤极端租金 (训练稳健化)
############################################
y_tr_raw = pd.to_numeric(df_tr_num[TARGET_COL], errors="coerce")
y_tr_raw = y_tr_raw.fillna(y_tr_raw.median())

Q1 = y_tr_raw.quantile(0.25)
Q3 = y_tr_raw.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR

mask_ok = (y_tr_raw >= lower_cut) & (y_tr_raw <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr_raw.loc[mask_ok].reset_index(drop=True)

print("训练样本(去极端后):", X_ok.shape)
print("测试样本:", X_test_final.shape)

############################################
# 6. 定义四个模型 (统一用pipeline+StandardScaler)
############################################
model_ols = make_pipeline(
    StandardScaler(with_mean=False),
    LinearRegression()
)

model_ridge = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(alpha=RIDGE_ALPHA_BEST,
          fit_intercept=True,
          random_state=42)
)

model_lasso = make_pipeline(
    StandardScaler(with_mean=False),
    Lasso(alpha=LASSO_ALPHA_BEST,
          fit_intercept=True,
          max_iter=50000,
          random_state=42)
)

model_enet = make_pipeline(
    StandardScaler(with_mean=False),
    ElasticNet(alpha=ENET_ALPHA_BEST,
               l1_ratio=ENET_L1RATIO_BEST,
               fit_intercept=True,
               max_iter=50000,
               random_state=42)
)

############################################
# 7. 训练四个模型
############################################
print("拟合 OLS...")
model_ols.fit(X_ok, y_ok)

print("拟合 Ridge...")
model_ridge.fit(X_ok, y_ok)

print("拟合 Lasso...")
model_lasso.fit(X_ok, y_ok)

print("拟合 ElasticNet...")
model_enet.fit(X_ok, y_ok)

############################################
# 8. 预测测试集
############################################
pred_ols_raw   = model_ols.predict(X_test_final)
pred_ridge_raw = model_ridge.predict(X_test_final)
pred_lasso_raw = model_lasso.predict(X_test_final)
pred_enet_raw  = model_enet.predict(X_test_final)

############################################
# 9. 修正负值（下界截断）
############################################
# 下界我用0，你要是觉得现实中最便宜也得>1000，就把0改成1000
floor_val = 0.0

pred_ols   = np.maximum(pred_ols_raw,   floor_val)
pred_ridge = np.maximum(pred_ridge_raw, floor_val)
pred_lasso = np.maximum(pred_lasso_raw, floor_val)
pred_enet  = np.maximum(pred_enet_raw,  floor_val)

############################################
# 10. sanity check 看看数量级
############################################
for name, arr in [
    ("OLS",   pred_ols),
    ("Ridge", pred_ridge),
    ("Lasso", pred_lasso),
    ("ENet",  pred_enet),
]:
    print(
        f"{name:>6} → min={np.min(arr):.2f}, "
        f"median={np.median(arr):.2f}, "
        f"max={np.max(arr):.2f}"
    )

############################################
# 11. 导出四个结果 (ID, Price) 每个模型一份
############################################
sub_ols = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_ols, 0).astype(int)
})
sub_ridge = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_ridge, 0).astype(int)
})
sub_lasso = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_lasso, 0).astype(int)
})
sub_enet = pd.DataFrame({
    "ID": df_te_raw["ID"].values,
    "Price": np.round(pred_enet, 0).astype(int)
})

sub_ols.to_excel(OUT_OLS,  index=False)
sub_ridge.to_excel(OUT_RID, index=False)
sub_lasso.to_excel(OUT_LAS, index=False)
sub_enet.to_excel(OUT_ENET, index=False)

# 如果你还想吐 csv
sub_ols.to_csv("rent_pred_OLS.csv", index=False)
sub_ridge.to_csv("rent_pred_Ridge.csv", index=False)
sub_lasso.to_csv("rent_pred_Lasso.csv", index=False)
sub_enet.to_csv("rent_pred_ElasticNet.csv", index=False)

print("导出完成（负值已截断）✅")



训练样本(去极端后): (93365, 140)
测试样本: (9773, 140)
拟合 OLS...
拟合 Ridge...
拟合 Lasso...
拟合 ElasticNet...
   OLS → min=0.00, median=378428.00, max=1993840.00
 Ridge → min=0.00, median=423739.08, max=2002146.80
 Lasso → min=0.00, median=436916.69, max=1851961.47
  ENet → min=0.00, median=447430.42, max=1180928.25
导出完成（负值已截断）✅


In [10]:
import numpy as np
import pandas as pd
import time

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============== 可配置区域（按需改） =================
train_path = "副本ruc_Class25Q2_train_rent91.csv"
test_path  = "要用租房的数据.csv"
TARGET_COL = "Price"
K_CLUSTERS = 80

# 你已有的“较优”超参（按你自己记录改）
RIDGE_ALPHA_BEST   = 100.0
LASSO_ALPHA_BEST   = 1000.0
ENET_ALPHA_BEST    = 10.0
ENET_L1RATIO_BEST  = 0.9

# 开启/关闭某些交互与非线性特征
USE_INTERACTIONS = True
USE_NONLINEAR_X  = True   # 对自变量做少量单调变换（不改 y 的刻度）

# 交叉验证、随机种子
KF_SPLITS    = 6
RANDOM_STATE = 111

# 结果导出
OUT_OLS  = "rent_pred_plus_OLS.xlsx"
OUT_RID  = "rent_pred_plus_Ridge.xlsx"
OUT_LAS  = "rent_pred_plus_Lasso.xlsx"
OUT_ENET = "rent_pred_plus_ElasticNet.xlsx"

# ============== 读数 + 基础清洗 =================
df_tr_raw = pd.read_csv(train_path)
df_te_raw = pd.read_csv(test_path)

if "ID" not in df_te_raw.columns:
    df_te_raw["ID"] = np.arange(len(df_te_raw)) + 1_000_000

# 修正可能存在的空格列名（你这批基本用不上，但留钩子）
rename_map = {"绿 化 率": "绿化率", "容 积 率": "容积率", "物 业 费": "物业费"}
df_tr_raw = df_tr_raw.rename(columns=rename_map)
df_te_raw = df_te_raw.rename(columns=rename_map)

# ============== 基础数值列 & 安全转换 =================
numeric_cols = [
    TARGET_COL,  # 训练集才有
    "面积","室","lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
]

def force_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    if "电梯_有" in out.columns:
        out["电梯_有"] = out["电梯_有"].fillna(0)
    return out

df_tr = force_numeric(df_tr_raw, numeric_cols)
df_te = force_numeric(df_te_raw, numeric_cols)

# ============== 非线性变换（只对自变量） =============
def add_nonlinear(df):
    out = df.copy()
    # 规模/密度类：log 或 sqrt（避免极端拉扯；保持单调）
    if "面积" in out.columns:
        out["log_面积"] = np.log1p(np.clip(out["面积"], 0, None))
        out["sqrt_面积"] = np.sqrt(np.clip(out["面积"], 0, None))
    if "房屋总数" in out.columns:
        out["log_房屋总数"] = np.log1p(np.clip(out["房屋总数"], 0, None))
    if "楼栋总数" in out.columns:
        out["log_楼栋总数"] = np.log1p(np.clip(out["楼栋总数"], 0, None))
    if "绿化率" in out.columns:
        out["sqrt_绿化率"] = np.sqrt(np.clip(out["绿化率"], 0, None))
    if "容积率" in out.columns:
        out["log_容积率"] = np.log1p(np.clip(out["容积率"], 0, None))
    if "物业费" in out.columns:
        out["log_物业费"] = np.log1p(np.clip(out["物业费"], 0, None))
    if "房龄" in out.columns:
        out["sqrt_房龄"] = np.sqrt(np.clip(out["房龄"], 0, None))
    return out

if USE_NONLINEAR_X:
    df_tr = add_nonlinear(df_tr)
    df_te = add_nonlinear(df_te)

# ============== 交互项（更“贴租房逻辑”的组合） =========
def add_interactions(df):
    out = df.copy()
    def safe_mul(a, b):
        return pd.to_numeric(out.get(a), errors="coerce") * pd.to_numeric(out.get(b), errors="coerce")

    # 面积 × 高层/中层/地下室（楼层影响使用价值 & 噪音/采光）
    if "面积" in out.columns:
        if "楼层_高层" in out.columns:
            out["交互_面积x高层"] = safe_mul("面积", "楼层_高层")
        if "楼层_中层" in out.columns:
            out["交互_面积x中层"] = safe_mul("面积", "楼层_中层")
        if "楼层_地下室" in out.columns:
            out["交互_面积x地下室"] = safe_mul("面积", "楼层_地下室")

    # 高层 × 电梯有（电梯在高层的边际价值更大）
    if "楼层_高层" in out.columns and "电梯_有" in out.columns:
        out["交互_高层x电梯"] = safe_mul("楼层_高层", "电梯_有")

    # 面积 × 电梯有（电梯对大面积户型更友好，入住体验增值）
    if "面积" in out.columns and "电梯_有" in out.columns:
        out["交互_面积x电梯"] = safe_mul("面积", "电梯_有")

    # 房龄 × 电梯有（老房有电梯 = 稀缺改善）
    if "房龄" in out.columns and "电梯_有" in out.columns:
        out["交互_房龄x电梯"] = safe_mul("房龄", "电梯_有")

    # 面积 × 付款方式（月付/季付/年付：流动性溢价/折价）
    for pay in ["付款方式_月付价","付款方式_季付价","付款方式_年付价","付款方式_双月付价"]:
        if pay in out.columns and "面积" in out.columns:
            out[f"交互_面积x{pay}"] = safe_mul("面积", pay)

    # 面积 × 绿化率；面积 × 容积率（居住舒适度与密度）
    if "面积" in out.columns and "绿化率" in out.columns:
        out["交互_面积x绿化率"] = safe_mul("面积", "绿化率")
    if "面积" in out.columns and "容积率" in out.columns:
        out["交互_面积x容积率"] = safe_mul("面积", "容积率")

    # 容积率 × 绿化率（社区密度-绿化的组合）
    if "容积率" in out.columns and "绿化率" in out.columns:
        out["交互_容积率x绿化率"] = safe_mul("容积率", "绿化率")

    # 设施总数（稀疏10个设施 → 汇总一个“配置评分”）
    facility_cols = [c for c in out.columns if c.startswith("设施_")]
    if facility_cols:
        out["设施_合计"] = out[facility_cols].sum(axis=1)

    # 非线性交互：log_面积 × 电梯（更稳）
    if "log_面积" in out.columns and "电梯_有" in out.columns:
        out["交互_log面积x电梯"] = safe_mul("log_面积", "电梯_有")

    return out

if USE_INTERACTIONS:
    df_tr = add_interactions(df_tr)
    df_te = add_interactions(df_te)

# ============== KMeans: market_cluster80（训练fit，测试predict） =========
cluster_basis = [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "电梯_有"
]
train_mat = df_tr[cluster_basis].copy()
for c in train_mat.columns:
    train_mat[c] = pd.to_numeric(train_mat[c], errors="coerce")
train_mat = train_mat.fillna(train_mat.median(numeric_only=True))

scaler_cluster = StandardScaler()
train_scaled = scaler_cluster.fit_transform(train_mat)

K_use = min(K_CLUSTERS, len(df_tr))
kmeans = KMeans(n_clusters=K_use, random_state=42, n_init=10)
df_tr["market_cluster80"] = kmeans.fit_predict(train_scaled).astype(int)

test_mat = df_te[cluster_basis].copy()
for c in test_mat.columns:
    test_mat[c] = pd.to_numeric(test_mat[c], errors="coerce")
test_mat = test_mat.fillna(train_mat.median(numeric_only=True))
test_scaled = scaler_cluster.transform(test_mat)
df_te["market_cluster80"] = kmeans.predict(test_scaled).astype(int)

# ============== 设计矩阵（主特征 + 城市dummy + cluster80 dummy + 新特征） =========
base_cols = [
    # 原主特征
    "面积","室","房龄","lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "楼层_中层","楼层_高层","楼层_地下室","电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
] + [
    # 非线性
    "log_面积","sqrt_面积","log_房屋总数","log_楼栋总数","sqrt_绿化率","log_容积率","log_物业费","sqrt_房龄",
    # 交互
    "交互_面积x高层","交互_面积x中层","交互_面积x地下室",
    "交互_高层x电梯","交互_面积x电梯","交互_房龄x电梯",
    "交互_面积x月付价","交互_面积x季付价","交互_面积x年付价","交互_面积x双月付价",
    "交互_面积x绿化率","交互_面积x容积率","交互_容积率x绿化率",
    "设施_合计","交互_log面积x电梯"
]

def build_X(df):
    df2 = df.copy()
    # 城市 dummy
    city_dum = pd.get_dummies(df2["城市"], prefix="city", drop_first=False) if "城市" in df2.columns else pd.DataFrame(index=df2.index)
    # cluster dummy
    clus_dum = pd.get_dummies(df2["market_cluster80"], prefix="cluster80", drop_first=False).astype(int)

    keep = [c for c in base_cols if c in df2.columns]
    X_main = df2[keep].copy()
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    X_full = pd.concat([X_main.reset_index(drop=True),
                        city_dum.reset_index(drop=True),
                        clus_dum.reset_index(drop=True)], axis=1)
    # 去常数列
    X_full = X_full.loc[:, X_full.std(axis=0) > 0]
    return X_full

X_tr_full = build_X(df_tr)
X_te_full = build_X(df_te)

# 训练列顺序 → 对齐测试集
train_cols = X_tr_full.columns
X_test_final = X_te_full.reindex(columns=train_cols, fill_value=0)

# ============== y & IQR 去极端值（只动训练集） =========
y_tr = pd.to_numeric(df_tr[TARGET_COL], errors="coerce").fillna(method="ffill")
Q1, Q3 = y_tr.quantile(0.25), y_tr.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_tr >= lower_cut) & (y_tr <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr.loc[mask_ok].reset_index(drop=True)

print(f"[INFO] 训练样本(去极端): {X_ok.shape}, 测试样本: {X_test_final.shape}, 特征数: {X_ok.shape[1]}")

# ============== 模型定义（同一套路，不耗时大调参） =========
model_ols = make_pipeline(StandardScaler(with_mean=False), LinearRegression())
model_ridge = make_pipeline(StandardScaler(with_mean=False),
                            Ridge(alpha=RIDGE_ALPHA_BEST, fit_intercept=True, random_state=42))
model_lasso = make_pipeline(StandardScaler(with_mean=False),
                            Lasso(alpha=LASSO_ALPHA_BEST, fit_intercept=True, max_iter=50000, random_state=42))
model_enet  = make_pipeline(StandardScaler(with_mean=False),
                            ElasticNet(alpha=ENET_ALPHA_BEST, l1_ratio=ENET_L1RATIO_BEST,
                                       fit_intercept=True, max_iter=50000, random_state=42))

from sklearn.base import clone

def eval_model(name, model, X, y, testX):
    print(f"\n===== 评估 {name} =====", flush=True)

    # 拟合全训练集
    t0 = time.time()
    model.fit(X, y)
    t1 = time.time()
    print(f"  拟合耗时: {t1 - t0:.2f}s")

    # 样本内
    p_tr = model.predict(X)
    tr_mae = mean_absolute_error(y, p_tr)
    tr_mse = mean_squared_error(y, p_tr)
    tr_rmse = np.sqrt(tr_mse)
    tr_r2 = r2_score(y, p_tr)
    print("  [训练集]  MAE={:,.2f} | RMSE={:,.2f} | R²={:.4f}".format(tr_mae, tr_rmse, tr_r2))

    # 6 折交叉验证
    kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_mae, cv_rmse, cv_r2 = [], [], []
    for i, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        # 关键：克隆同结构未拟合的 Pipeline，避免复用已拟合对象
        m = clone(model)
        m.fit(X_tr, y_tr_)
        pv = m.predict(X_va)

        mae = mean_absolute_error(y_va, pv)
        rmse = np.sqrt(mean_squared_error(y_va, pv))
        r2 = r2_score(y_va, pv)

        cv_mae.append(mae)
        cv_rmse.append(rmse)
        cv_r2.append(r2)

    print("  [6折CV]  MAE={:,.2f}±{:.2f} | RMSE={:,.2f}±{:.2f} | R²={:.4f}±{:.4f}".format(
        np.mean(cv_mae), np.std(cv_mae), np.mean(cv_rmse), np.std(cv_rmse), np.mean(cv_r2), np.std(cv_r2)
    ))

    # 测试集预测（负值截断）
    p_te_raw = model.predict(testX)
    p_te = np.maximum(p_te_raw, 0.0)
    print("  [测试预测] min={:.2f} | median={:.2f} | max={:.2f}".format(
        float(np.min(p_te)), float(np.median(p_te)), float(np.max(p_te))
    ))

    return p_te


# ============== 开始评估四个模型（只换特征，不大调参） =========
pred_ols   = eval_model("OLS",        model_ols,   X_ok, y_ok, X_test_final)
pred_ridge = eval_model("Ridge(L2)",  model_ridge, X_ok, y_ok, X_test_final)
pred_lasso = eval_model("Lasso(L1)",  model_lasso, X_ok, y_ok, X_test_final)
pred_enet  = eval_model("ElasticNet", model_enet,  X_ok, y_ok, X_test_final)

# ============== 导出四份 Excel（ID, Price） =========
def to_xlsx(path, preds):
    sub = pd.DataFrame({"ID": df_te_raw["ID"].values, "Price": np.round(preds, 0).astype(int)})
    sub.to_excel(path, index=False)

to_xlsx(OUT_OLS,  pred_ols)
to_xlsx(OUT_RID,  pred_ridge)
to_xlsx(OUT_LAS,  pred_lasso)
to_xlsx(OUT_ENET, pred_enet)
print("\n导出完成（新增交互/非线性已生效）。")


[INFO] 训练样本(去极端): (93365, 146), 测试样本: (9773, 146), 特征数: 146

===== 评估 OLS =====


/var/folders/xj/4k974rbd7g1dm1sw08fyntg00000gn/T/ipykernel_16631/1454432061.py:238: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  y_tr = pd.to_numeric(df_tr[TARGET_COL], errors="coerce").fillna(method="ffill")


  拟合耗时: 0.82s
  [训练集]  MAE=114,540.74 | RMSE=157,992.75 | R²=0.7284
  [6折CV]  MAE=114,776.25±473.70 | RMSE=158,314.05±669.76 | R²=0.7273±0.0029
  [测试预测] min=0.00 | median=394016.00 | max=1430968.00

===== 评估 Ridge(L2) =====
  拟合耗时: 0.33s
  [训练集]  MAE=115,967.26 | RMSE=159,560.09 | R²=0.7230
  [6折CV]  MAE=116,303.44±465.16 | RMSE=159,989.82±632.82 | R²=0.7215±0.0027
  [测试预测] min=0.00 | median=436375.54 | max=1436776.96

===== 评估 Lasso(L1) =====
  拟合耗时: 8.30s
  [训练集]  MAE=117,059.19 | RMSE=161,237.40 | R²=0.7171
  [6折CV]  MAE=117,233.08±416.34 | RMSE=161,481.57±571.00 | R²=0.7162±0.0023
  [测试预测] min=0.00 | median=448323.83 | max=5883811.48

===== 评估 ElasticNet =====
  拟合耗时: 0.93s
  [训练集]  MAE=138,727.95 | RMSE=187,761.63 | R²=0.6164
  [6折CV]  MAE=138,821.35±669.54 | RMSE=187,890.61±819.33 | R²=0.6159±0.0005
  [测试预测] min=10386.83 | median=465678.54 | max=8261072.49

导出完成（新增交互/非线性已生效）。


In [12]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

# ===================== 配置区 =====================
train_path = "副本ruc_Class25Q2_train_rent91.csv"
test_path  = "要用租房的数据.csv"
TARGET_COL = "Price"               # 训练集里的租金列名
K_CLUSTERS = 80

# 你当前较优的超参（按你的结果改）
RIDGE_ALPHA_BEST  = 100.0
LASSO_ALPHA_BEST  = 1000.0
ENET_ALPHA_BEST   = 100.0          # 若你另有更优值，改这里
ENET_L1RATIO_BEST = 0.5            # 0~1 之间，越大越偏L1

# 交互/非线性特征开关
USE_INTERACTIONS = True
USE_NONLINEAR_X  = True

# 交叉验证/随机种子
KF_SPLITS    = 6
RANDOM_STATE = 111

# 导出文件名
OUT_OLS  = "rent_pred_OLS.xlsx"
OUT_RID  = "rent_pred_Ridge.xlsx"
OUT_LAS  = "rent_pred_Lasso.xlsx"
OUT_ENET = "rent_pred_ElasticNet.xlsx"

# ===================== 工具函数 =====================
def force_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    if "电梯_有" in out.columns:
        out["电梯_有"] = out["电梯_有"].fillna(0)
    return out

def add_nonlinear(df):
    out = df.copy()
    def clip0(s): return np.clip(pd.to_numeric(s, errors="coerce"), 0, None)
    if "面积" in out.columns:
        out["log_面积"]  = np.log1p(clip0(out["面积"]))
        out["sqrt_面积"] = np.sqrt(clip0(out["面积"]))
    if "房屋总数" in out.columns:
        out["log_房屋总数"] = np.log1p(clip0(out["房屋总数"]))
    if "楼栋总数" in out.columns:
        out["log_楼栋总数"] = np.log1p(clip0(out["楼栋总数"]))
    if "绿化率" in out.columns:
        out["sqrt_绿化率"] = np.sqrt(clip0(out["绿化率"]))
    if "容积率" in out.columns:
        out["log_容积率"] = np.log1p(clip0(out["容积率"]))
    if "物业费" in out.columns:
        out["log_物业费"] = np.log1p(clip0(out["物业费"]))
    if "房龄" in out.columns:
        out["sqrt_房龄"] = np.sqrt(clip0(out["房龄"]))
    return out

def add_interactions(df):
    out = df.copy()
    def mul(a, b):
        return pd.to_numeric(out.get(a), errors="coerce") * pd.to_numeric(out.get(b), errors="coerce")

    # 面积 × 楼层
    if "面积" in out.columns:
        if "楼层_高层" in out.columns:
            out["交互_面积x高层"] = mul("面积","楼层_高层")
        if "楼层_中层" in out.columns:
            out["交互_面积x中层"] = mul("面积","楼层_中层")
        if "楼层_地下室" in out.columns:
            out["交互_面积x地下室"] = mul("面积","楼层_地下室")

    # 楼层 × 电梯；面积 × 电梯；房龄 × 电梯
    if "楼层_高层" in out.columns and "电梯_有" in out.columns:
        out["交互_高层x电梯"] = mul("楼层_高层","电梯_有")
    if "面积" in out.columns and "电梯_有" in out.columns:
        out["交互_面积x电梯"] = mul("面积","电梯_有")
    if "房龄" in out.columns and "电梯_有" in out.columns:
        out["交互_房龄x电梯"] = mul("房龄","电梯_有")

    # 面积 × 付款方式
    for pay in ["付款方式_月付价","付款方式_季付价","付款方式_年付价","付款方式_双月付价"]:
        if pay in out.columns and "面积" in out.columns:
            out[f"交互_面积x{pay}"] = mul("面积", pay)

    # 面积 × 绿化率/容积率；容积率 × 绿化率
    if "面积" in out.columns and "绿化率" in out.columns:
        out["交互_面积x绿化率"] = mul("面积","绿化率")
    if "面积" in out.columns and "容积率" in out.columns:
        out["交互_面积x容积率"] = mul("面积","容积率")
    if "容积率" in out.columns and "绿化率" in out.columns:
        out["交互_容积率x绿化率"] = mul("容积率","绿化率")

    # 设施合计
    fac_cols = [c for c in out.columns if c.startswith("设施_")]
    if fac_cols:
        out["设施_合计"] = out[fac_cols].sum(axis=1)

    # 非线性交互：log(面积) × 电梯
    if "log_面积" in out.columns and "电梯_有" in out.columns:
        out["交互_log面积x电梯"] = mul("log_面积","电梯_有")

    return out

def build_X(df, base_cols):
    df2 = df.copy()
    city_dum = pd.get_dummies(df2["城市"], prefix="city", drop_first=False) if "城市" in df2.columns else pd.DataFrame(index=df2.index)
    clus_dum = pd.get_dummies(df2["market_cluster80"], prefix="cluster80", drop_first=False).astype(int)

    keep = [c for c in base_cols if c in df2.columns]
    X_main = df2[keep].copy()
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    X_full = pd.concat([X_main.reset_index(drop=True),
                        city_dum.reset_index(drop=True),
                        clus_dum.reset_index(drop=True)], axis=1)
    X_full = X_full.loc[:, X_full.std(axis=0) > 0]  # 去常数列
    return X_full

def eval_model(name, model, X, y, testX):
    print(f"\n===== 评估 {name} =====", flush=True)

    # 拟合全训练集
    t0 = time.time()
    model.fit(X, y)
    t1 = time.time()
    print(f"  拟合耗时: {t1 - t0:.2f}s")

    # 样本内
    p_tr = model.predict(X)
    tr_mae = mean_absolute_error(y, p_tr)
    tr_rmse = np.sqrt(mean_squared_error(y, p_tr))
    tr_r2 = r2_score(y, p_tr)
    print("  [训练集]  MAE={:,.2f} | RMSE={:,.2f} | R²={:.4f}".format(tr_mae, tr_rmse, tr_r2))

    # 6 折交叉验证
    kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_mae, cv_rmse, cv_r2 = [], [], []
    for i, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        m = clone(model)  # 克隆未拟合的同结构Pipeline
        m.fit(X_tr, y_tr_)
        pv = m.predict(X_va)
        cv_mae.append(mean_absolute_error(y_va, pv))
        cv_rmse.append(np.sqrt(mean_squared_error(y_va, pv)))
        cv_r2.append(r2_score(y_va, pv))

    print("  [6折CV]  MAE={:,.2f}±{:.2f} | RMSE={:,.2f}±{:.2f} | R²={:.4f}±{:.4f}".format(
        np.mean(cv_mae), np.std(cv_mae), np.mean(cv_rmse), np.std(cv_rmse), np.mean(cv_r2), np.std(cv_r2)
    ))

    # 测试集预测（先不截断，交给分组回填函数处理负值）
    p_te_raw = model.predict(testX)
    return p_te_raw

def build_group_stats(df_tr, mask_ok, y_ok,
                      group_cols=("城市","market_cluster80"),
                      city_col="城市"):
    use_cols = [c for c in group_cols if c in df_tr.columns]
    grp_df = pd.DataFrame(index=y_ok.index)
    for c in use_cols:
        grp_df[c] = df_tr.loc[mask_ok, c].reset_index(drop=True)
    grp_df["Price"] = y_ok.values

    if len(use_cols) >= 2:
        pair_median = grp_df.groupby(use_cols)["Price"].median()
        pair_dict = {k: v for k, v in pair_median.items()}
    else:
        pair_dict = {}

    city_dict = {}
    if city_col in grp_df.columns:
        city_median = grp_df.groupby(city_col)["Price"].median()
        city_dict = city_median.to_dict()

    global_median = float(grp_df["Price"].median())
    return pair_dict, city_dict, global_median, use_cols

def fill_negative_with_group(pred, df_te, pair_dict, city_dict, global_median, use_cols):
    pred = np.asarray(pred).astype(float)
    neg_idx = pred < 0
    if not np.any(neg_idx):
        return pred, 0

    # (城市,cluster) 键
    keys = []
    for i in range(len(df_te)):
        if len(use_cols) >= 2 and all(c in df_te.columns for c in use_cols):
            key_tuple = tuple(df_te.iloc[i][use_cols].tolist())
        else:
            key_tuple = None
        keys.append(key_tuple)

    # 城市列
    city_vals = df_te[use_cols[0]].values if use_cols and use_cols[0] in df_te.columns else np.array([None]*len(df_te))

    filled = pred.copy()
    replaced = 0
    for i in np.where(neg_idx)[0]:
        val = None
        if keys[i] is not None and keys[i] in pair_dict:
            val = pair_dict[keys[i]]
        if val is None and city_vals[i] in city_dict:
            val = city_dict[city_vals[i]]
        if val is None:
            val = global_median
        filled[i] = val
        replaced += 1
    return filled, replaced

def to_xlsx(path, ids, preds):
    sub = pd.DataFrame({"ID": ids, "Price": np.round(preds, 0).astype(int)})
    sub.to_excel(path, index=False)

# ===================== 主流程 =====================
# 读数据
df_tr_raw = pd.read_csv(train_path)
df_te_raw = pd.read_csv(test_path)

if "ID" not in df_te_raw.columns:
    df_te_raw["ID"] = np.arange(len(df_te_raw)) + 2_000_000

# 变量名对齐（去空格）
rename_map = {"绿 化 率":"绿化率", "容 积 率":"容积率", "物 业 费":"物业费"}
df_tr_raw = df_tr_raw.rename(columns=rename_map)
df_te_raw = df_te_raw.rename(columns=rename_map)

# 数值列转换
numeric_cols = [
    TARGET_COL, "面积","室","lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
]
df_tr = force_numeric(df_tr_raw, numeric_cols)
df_te = force_numeric(df_te_raw, numeric_cols)

# 非线性/交互
if USE_NONLINEAR_X:
    df_tr = add_nonlinear(df_tr)
    df_te = add_nonlinear(df_te)
if USE_INTERACTIONS:
    df_tr = add_interactions(df_tr)
    df_te = add_interactions(df_te)

# KMeans: market_cluster80（训练fit，测试predict）
cluster_basis = ["lon","lat","房龄","绿化率","容积率","物业费","房屋总数","楼栋总数","电梯_有"]
train_mat = df_tr[cluster_basis].copy()
for c in train_mat.columns:
    train_mat[c] = pd.to_numeric(train_mat[c], errors="coerce")
train_mat = train_mat.fillna(train_mat.median(numeric_only=True))

scaler_cluster = StandardScaler()
train_scaled = scaler_cluster.fit_transform(train_mat)

K_use = min(K_CLUSTERS, len(df_tr))
kmeans = KMeans(n_clusters=K_use, random_state=42, n_init=10)
df_tr["market_cluster80"] = kmeans.fit_predict(train_scaled).astype(int)

test_mat = df_te[cluster_basis].copy()
for c in test_mat.columns:
    test_mat[c] = pd.to_numeric(test_mat[c], errors="coerce")
test_mat = test_mat.fillna(train_mat.median(numeric_only=True))
test_scaled = scaler_cluster.transform(test_mat)
df_te["market_cluster80"] = kmeans.predict(test_scaled).astype(int)

# 设计矩阵
base_cols = [
    "面积","室","房龄","lon","lat",
    "房屋总数","楼栋总数","绿化率","容积率","物业费",
    "楼层_中层","楼层_高层","楼层_地下室","电梯_有",
    "租赁方式_整租",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    # 非线性
    "log_面积","sqrt_面积","log_房屋总数","log_楼栋总数","sqrt_绿化率","log_容积率","log_物业费","sqrt_房龄",
    # 交互
    "交互_面积x高层","交互_面积x中层","交互_面积x地下室",
    "交互_高层x电梯","交互_面积x电梯","交互_房龄x电梯",
    "交互_面积x月付价","交互_面积x季付价","交互_面积x年付价","交互_面积x双月付价",
    "交互_面积x绿化率","交互_面积x容积率","交互_容积率x绿化率",
    "设施_合计","交互_log面积x电梯"
]
X_tr_full = build_X(df_tr, base_cols)
X_te_full = build_X(df_te,  base_cols)

# 列对齐
train_cols = X_tr_full.columns
X_test_final = X_te_full.reindex(columns=train_cols, fill_value=0)

# y 与 IQR 去极端（仅训练集）
y_tr = pd.to_numeric(df_tr[TARGET_COL], errors="coerce")
keep_y = y_tr.notna()
X_tr_full = X_tr_full.loc[keep_y].reset_index(drop=True)
df_tr = df_tr.loc[keep_y].reset_index(drop=True)
y_tr = y_tr.loc[keep_y].reset_index(drop=True)

Q1, Q3 = y_tr.quantile(0.25), y_tr.quantile(0.75)
IQR = Q3 - Q1
lower_cut, upper_cut = Q1 - 1.5*IQR, Q3 + 1.5*IQR
mask_ok = (y_tr >= lower_cut) & (y_tr <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr.loc[mask_ok].reset_index(drop=True)

print(f"[INFO] 训练样本(去极端): {X_ok.shape}, 测试样本: {X_test_final.shape}, 特征数: {X_ok.shape[1]}")

# 模型
model_ols   = make_pipeline(StandardScaler(with_mean=False), LinearRegression())
model_ridge = make_pipeline(StandardScaler(with_mean=False), Ridge(alpha=RIDGE_ALPHA_BEST, fit_intercept=True, random_state=42))
model_lasso = make_pipeline(StandardScaler(with_mean=False), Lasso(alpha=LASSO_ALPHA_BEST, fit_intercept=True, max_iter=50000, random_state=42))
model_enet  = make_pipeline(StandardScaler(with_mean=False), ElasticNet(alpha=ENET_ALPHA_BEST, l1_ratio=ENET_L1RATIO_BEST,
                                                                        fit_intercept=True, max_iter=50000, random_state=42))

# 训练 + 预测（先拿 raw 预测）
pred_ols_raw   = eval_model("OLS",        model_ols,   X_ok, y_ok, X_test_final)
pred_ridge_raw = eval_model("Ridge(L2)",  model_ridge, X_ok, y_ok, X_test_final)
pred_lasso_raw = eval_model("Lasso(L1)",  model_lasso, X_ok, y_ok, X_test_final)
pred_enet_raw  = eval_model("ElasticNet", model_enet,  X_ok, y_ok, X_test_final)

# 分组中位数回填负值
pair_dict, city_dict, global_med, used_cols = build_group_stats(
    df_tr=df_tr, mask_ok=mask_ok, y_ok=y_ok,
    group_cols=("城市","market_cluster80"), city_col="城市"
)

pred_ols,   rep_ols   = fill_negative_with_group(pred_ols_raw,   df_te_raw, pair_dict, city_dict, global_med, used_cols)
pred_ridge, rep_ridge = fill_negative_with_group(pred_ridge_raw, df_te_raw, pair_dict, city_dict, global_med, used_cols)
pred_lasso, rep_lasso = fill_negative_with_group(pred_lasso_raw, df_te_raw, pair_dict, city_dict, global_med, used_cols)
pred_enet,  rep_enet  = fill_negative_with_group(pred_enet_raw,  df_te_raw, pair_dict, city_dict, global_med, used_cols)

print(f"[负值回填条数] OLS={rep_ols} | Ridge={rep_ridge} | Lasso={rep_lasso} | ENet={rep_enet}")
print(f"[组统计] used_cols={used_cols}, global_median={global_med:,.0f}")

# 导出
to_xlsx(OUT_OLS,  df_te_raw["ID"].values, pred_ols)
to_xlsx(OUT_RID,  df_te_raw["ID"].values, pred_ridge)
to_xlsx(OUT_LAS,  df_te_raw["ID"].values, pred_lasso)
to_xlsx(OUT_ENET, df_te_raw["ID"].values, pred_enet)
print("\n✅ 已导出四个Excel：", OUT_OLS, OUT_RID, OUT_LAS, OUT_ENET)


[INFO] 训练样本(去极端): (93365, 146), 测试样本: (9773, 146), 特征数: 146

===== 评估 OLS =====
  拟合耗时: 0.71s
  [训练集]  MAE=114,540.74 | RMSE=157,992.75 | R²=0.7284
  [6折CV]  MAE=114,776.25±473.70 | RMSE=158,314.05±669.76 | R²=0.7273±0.0029

===== 评估 Ridge(L2) =====
  拟合耗时: 0.31s
  [训练集]  MAE=115,967.26 | RMSE=159,560.09 | R²=0.7230
  [6折CV]  MAE=116,303.44±465.16 | RMSE=159,989.82±632.82 | R²=0.7215±0.0027

===== 评估 Lasso(L1) =====
  拟合耗时: 6.97s
  [训练集]  MAE=117,059.19 | RMSE=161,237.40 | R²=0.7171
  [6折CV]  MAE=117,233.08±416.34 | RMSE=161,481.57±571.00 | R²=0.7162±0.0023

===== 评估 ElasticNet =====
  拟合耗时: 0.42s
  [训练集]  MAE=239,492.00 | RMSE=293,487.39 | R²=0.0628
  [6折CV]  MAE=239,497.59±1199.21 | RMSE=293,491.26±1429.78 | R²=0.0627±0.0003
[负值回填条数] OLS=1019 | Ridge=123 | Lasso=20 | ENet=0
[组统计] used_cols=['城市', 'market_cluster80'], global_median=368,758

✅ 已导出四个Excel： rent_pred_OLS.xlsx rent_pred_Ridge.xlsx rent_pred_Lasso.xlsx rent_pred_ElasticNet.xlsx
